# Данные, токены и маски

Как подготовить текст и не обучаться на padding.

> Все примеры этого урока работают на CPU, не скачивают модели и используют искусственные данные. Они объясняют механизм; качество настоящего обучения проверяется отдельно.

## Что важно понять

Батч содержит последовательности разной длины. Padding заполняет пустые позиции, attention mask запрещает использовать их как контекст, а labels=-100 исключает их из cross entropy. Это разные механизмы.

Для SFT важно сохранять границы ролей и отвечать на вопрос, какие токены участвуют в loss. Для изображений, действий и эпизодов дополнительно нужны согласованные размеры, единицы и временные границы. Не смешивайте кадры одного эпизода между train и validation.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Запустите эксперимент

Запускайте ячейки по порядку. Код можно менять; Shift + Enter выполняет текущую ячейку.

In [ ]:
samples = ['model', 'learn']
tokens = [list(s.encode('utf-8')) for s in samples]
width = max(map(len, tokens)) + 3
ids = torch.zeros(len(tokens), width, dtype=torch.long)
mask = torch.zeros_like(ids)
for row, sequence in enumerate(tokens):
    ids[row, :len(sequence)] = torch.tensor(sequence)
    mask[row, :len(sequence)] = 1
labels = ids.clone(); labels[mask == 0] = -100
logits = torch.randn(*ids.shape, 256)
loss = F.cross_entropy(logits.flatten(0, 1), labels.flatten(), ignore_index=-100)
print('input_ids:', ids.tolist()); print('attention_mask:', mask.tolist())
print('labels:', labels.tolist()); print('masked loss:', loss.item())
assert (labels == -100).sum() == (mask == 0).sum()

## Попробуйте сами

Добавьте русский текст и посмотрите, почему число UTF-8 байтов отличается от числа символов. Это учебный byte tokenizer; настоящие модели используют свой tokenizer и chat template.

Сохраните копию ноутбука и запишите, что изменили, чего ожидали и что измерили.

## Продолжить в студии

Кнопка «Открыть раздел студии» над уроком ведёт к связанным инструментам. Учебный код сам не запускает большие тренировки. Перед реальным запуском проверьте данные, ресурсы и поддержку метода у выбранной модели.

## Источники

- [PyTorch: CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)